# Dataset Preparation

**Prerequisites**
- `.env` at repo root with `XC_API_KEY=your_key`
- `uv sync` run in `building/`

In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Parameters

In [10]:
N = 10
MAX_RECORDINGS = 10000
CLIPS_PER_SPECIES = 2500
MIN_XC_RECORDINGS = 100
BIRDNET_THRESHOLD = 0.92

ACTIVE_COLLECTIONS = ["diff_genus"] # ["diff_family", "diff_genus", "diff_species"]

# Select species

In [11]:
from building.taxonomy import (
    get_species_with_recordings,
    select_same_genus,
    select_same_family,
    select_same_order,
)

pool = get_species_with_recordings(min_recordings=MIN_XC_RECORDINGS)
print(f"Species pool: {len(pool)} species with {MIN_XC_RECORDINGS} XC recordings")

Species pool: 1805 species with 100 XC recordings


In [12]:
# show what you can choose (based on MIN_XC_RECORDINGS and your N)
from building.taxonomy import get_potential_taxa
print("Potential genera (need >=N species):")
print(get_potential_taxa("genus", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential families (need >=N distinct genera):")
print(get_potential_taxa("family", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))

print("\nPotential orders (need >=N distinct families):")
print(get_potential_taxa("order", n=N, min_recordings=MIN_XC_RECORDINGS, species_pool=pool).to_string(index=False))


Potential genera (need >=N species):
       genus  n_species  n_distinct_genus  n_distinct_family  n_distinct_order
Phylloscopus         32                 1                  1                 1
      Turdus         29                 1                  1                 1
   Setophaga         23                 1                  1                 1
    Emberiza         17                 1                  1                 1
       Vireo         15                 1                  1                 1
      Anthus         15                 1                  1                 1
Thamnophilus         14                 1                  1                 1
     Curruca         14                 1                  1                 1
Acrocephalus         12                 1                  1                 1
    Calidris         11                 1                  1                 1
       Strix         11                 1                  1                 1
      Trogon   

In [13]:
TARGET_GENUS  = "Phylloscopus"
TARGET_FAMILY = "Tyrannidae (Tyrant Flycatchers)"
TARGET_ORDER  = "Passeriformes"

In [14]:
collections: dict[str, list[str]] = {}
diff_species = select_same_genus(TARGET_GENUS, N, pool)
diff_genus = select_same_family(TARGET_FAMILY, N, pool)
diff_family = select_same_order(TARGET_ORDER, N, pool)
if "diff_species" in ACTIVE_COLLECTIONS:
    collections["diff_species"] = [s.scientific_name for s in diff_species]
if "diff_genus" in ACTIVE_COLLECTIONS:
    collections["diff_genus"] = [s.scientific_name for s in diff_genus]
if "diff_family" in ACTIVE_COLLECTIONS:
    collections["diff_family"] = [s.scientific_name for s in diff_family]

collections_to_use = {i_collection: collections[i_collection] for i_collection in ACTIVE_COLLECTIONS}

for coll, names in collections.items():
    print(f"\n{coll}:")
    for n in names:
        print(f"  {n}")
print(f"\nACTIVE_COLLECTIONS (download + BirdNET): {ACTIVE_COLLECTIONS}")


diff_genus:
  Pitangus sulphuratus
  Myiarchus tyrannulus
  Tolmomyias sulphurescens
  Tyrannus melancholicus
  Camptostoma obsoletum
  Attila spadiceus
  Megarynchus pitangua
  Myiozetetes similis
  Lathrotriccus euleri
  Myiodynastes maculatus

ACTIVE_COLLECTIONS (download + BirdNET): ['diff_genus']


# Download from Xeno-canto

In [ ]:
from building.download import download_and_process

for collection_name, species_names in collections.items():
    await download_and_process(species_names, 
        collection_name, 
        clips_per_species=CLIPS_PER_SPECIES, 
        max_recordings=MAX_RECORDINGS,
        auto_delete=True,)

Fetching available listings...
[Pitangus sulphuratus] available=359 processed=359 queued=0
[Myiarchus tyrannulus] available=343 processed=152 queued=191
[Tolmomyias sulphurescens] available=284 processed=227 queued=57
[Tyrannus melancholicus] available=241 processed=214 queued=27
[Camptostoma obsoletum] available=205 processed=0 queued=205
[Attila spadiceus] available=199 processed=62 queued=137
[Megarynchus pitangua] available=202 processed=196 queued=6
[Myiozetetes similis] available=191 processed=24 queued=167
[Lathrotriccus euleri] available=154 processed=0 queued=154
[Myiodynastes maculatus] available=159 processed=0 queued=159


Download+BirdNET:   0%|          | 0/1103 [00:00<?, ?it/s]

# BirdNET validation + dataset assembly

In [ ]:
from building.dataset import build_dataset

MAX_PER_CLASS = 1750
dataset_paths = {}
for coll_name, species_names in collections_to_use.items():
    dataset_paths[coll_name] = build_dataset(coll_name, species_names, clips_per_species=CLIPS_PER_SPECIES, max_class_size_training=MAX_PER_CLASS)
    print(f"  {coll_name}: {dataset_paths[coll_name]}")

FileExistsError: [Errno 17] File exists: '/home/nathan/Documents/multi-chirp/raw_dataset/subsamples/diff_genus/non_target_empty/clip_00229.wav' -> '/home/nathan/Documents/multi-chirp/datasets/diff_genus/training/non_target/clip_00229.wav'

# Sanity check

In [ ]:

for coll_name, root in dataset_paths.items():
    print(f"\n{coll_name}")
    for split in ("training", "validation", "testing"):
        split_dir = root / split
        if not split_dir.exists():
            continue
        species_counts = {
            d.name: len(list(d.glob("*.wav")))
            for d in sorted(split_dir.iterdir()) if d.is_dir()
        }
        total = sum(species_counts.values())
        print(f"  {split:12s} {total:5d} clips")
        for species, count in sorted(species_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"    {species:20s} {count:5d}")